# [9665] Latent Dirichlet Allocation 2

Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Hulu_titles.csv

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 02/23/25 19:51:46


### Import libraries

In [ ]:
import pandas as pd
import nltk
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer
from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim.models import LdaMulticore     # faster implementation of LDA (parallelized for multicore machines)
from gensim.models.coherencemodel import CoherenceModel
import multiprocessing

In [ ]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

### Load data

In [ ]:
pd.set_option('max_colwidth', None)

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Hulu_titles.csv')

### Examine data

In [ ]:
df.shape

(3073, 12)

In [ ]:
df.head(2)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Ricky Velez: Here's Everything,NaN,NaN,NaN,"October 24, 2021",2021,TV-MA,NaN,"Comedy, Stand Up",​Comedian Ricky Velez bares it all with his honest lens and down to earth perspective in his first-ever HBO stand-up special.
1,s2,Movie,Silent Night,NaN,NaN,NaN,"October 23, 2021",2020,NaN,94 min,"Crime, Drama, Thriller","Mark, a low end South London hitman recently released from prison, tries to go straight for his daughter, but gets drawn back in by Alan, his former cellmate, to do one final job."


Choose 3 movie title as examples for later
* Second Chance
* Little Birds
* Hawthorne

In [ ]:
# Some sample movie titles for later examination
SAMPLE_TITLE_1 = 'Second Chance'
SAMPLE_TITLE_2 = 'Little Birds'
SAMPLE_TITLE_3 = 'Hawthorne'

In [ ]:
# Get index values for sample movies
SAMPLE_TITLE_1_IDX = df[df['title'] == SAMPLE_TITLE_1].index.tolist()[0]
SAMPLE_TITLE_2_IDX = df[df['title'] == SAMPLE_TITLE_2].index.tolist()[0]
SAMPLE_TITLE_3_IDX = df[df['title'] == SAMPLE_TITLE_3].index.tolist()[0]

In [ ]:
# Create new dataframe with only columns 'title' amd 'description'
df2 = df[['title', 'description']].astype(str)
df2.head()

,title,description
0,Ricky Velez: Here's Everything,​Comedian Ricky Velez bares it all with his honest lens and down to earth perspective in his first-ever HBO stand-up special.
1,Silent Night,"Mark, a low end South London hitman recently released from prison, tries to go straight for his daughter, but gets drawn back in by Alan, his former cellmate, to do one final job."
2,The Marksman,A hardened Arizona rancher tries to protect an 11-year-old migrant boy fleeing from a ruthless drug cartel.
3,Gaia,A forest ranger and two survivalists with a cultish devotion to the forest face the threat of an unrelenting wilderness when a strange being attacks one night.
4,Settlers,Mankind's earliest settlers on the Martian frontier do what they must to survive the cosmic elements and each other in this science-fiction thrill ride.


### Preprocess data

In [ ]:
# Create preprocessing functions
def lemmatizer(text):
    return WordNetLemmatizer().lemmatize(text, pos='v')

def preprocess(text):
    result = []
    for token in simple_preprocess(text):
        if token not in STOPWORDS and len(token) > 3:
            result.append(lemmatizer(token))
    return result

In [ ]:
# Show results of stemmer for sample document
example_doc = df2.iloc[SAMPLE_TITLE_1_IDX, 1]
print('title: {}\n'.format(SAMPLE_TITLE_1))
print('original document:\n\t{}'.format(example_doc))
print('\ntokenized and lemmatized document:\n\t{}'.format(preprocess(example_doc)))

title: Second Chance

original document:
	What would you do with a second chance? From executive producer/writer Rand Ravich and Emmy Award-winning executive producer Howard Gordon comes SECOND CHANCE, a thrilling new action-drama about a man brought back to life by two scientists playing god in the quest to save one of their own lives. Seventy-five-year-old JIMMY PRITCHARD Philip Baker Hall) is a shell of his former self. A drinker, a womanizer and a father who always put work before family, Pritchard was forced to resign as L.A. County Sheriff for corrupt conduct more than a decade ago. Now, some 15 unkind years later, he is killed when he stumbles upon a robbery at the home of FBI Agent DUVAL PRITCHARD (Tim DeKay), one of his two children. But death is surprisingly short for Jimmy, who is brought back to life by billionaire tech-genius twins MARY GOODWIN (Dilshad Vadsaria) and her brother, OTTO (Adhir Kalyan), founders of a social networking empire. Resurrected as a younger, better 

In [ ]:
%%time

# Preprocess all documents
processed_docs = df2['description'].map(preprocess)

CPU times: user 777 ms, sys: 15.5 ms, total: 793 ms
Wall time: 816 ms


In [ ]:
# Review first few processed documents
processed_docs.head()

,description
0,"[comedian, ricky, velez, bar, honest, lens, earth, perspective, stand, special]"
1,"[mark, south, london, hitman, recently, release, prison, try, straight, daughter, get, draw, alan, cellmate, final]"
2,"[harden, arizona, rancher, try, protect, year, migrant, flee, ruthless, drug, cartel]"
3,"[forest, ranger, survivalists, cultish, devotion, forest, face, threat, unrelenting, wilderness, strange, attack, night]"
4,"[mankind, earliest, settlers, martian, frontier, survive, cosmic, elements, science, fiction, thrill, ride]"


### Generate Gensim Dictionary object

In [ ]:
%%time

# Map each word in ‘processed_docs’ to its unique integer id (index)
dictionary = Dictionary(processed_docs)

CPU times: user 110 ms, sys: 183 µs, total: 111 ms
Wall time: 115 ms


In [ ]:
# Display # of words in Dictionary object
print("{} words in Dictionary object".format(len(dictionary)))

13859 words in Dictionary object


In [ ]:
# Display first 10 elements in dictionary
count = 0
for k, v in dictionary.iteritems():
    print(k, v)
    count += 1
    if count > 10:
        break

0 bar
1 comedian
2 earth
3 honest
4 lens
5 perspective
6 ricky
7 special
8 stand
9 velez
10 alan


In [ ]:
# Remove very rare and very common words
#  Filter out tokens that appear in
#   < 15 documents (absolute number) or
#   > 50% documents (fraction of total corpus size, not absolute number)
#  After the above two steps, keep only the first 100000 most frequent tokens
dictionary.filter_extremes(no_below=15, no_above=0.5, keep_n=100000)
print("{} words remaining in Dictionary object".format(len(dictionary)))

817 words remaining in Dictionary object


In [ ]:
%%time

# Convert documents into the bag-of-words format: list of (token_id, token_count) 2-tuples
bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

CPU times: user 58.7 ms, sys: 2.79 ms, total: 61.5 ms
Wall time: 78.5 ms


In [ ]:
# Check processed BoW version of sample document
print(bow_corpus[SAMPLE_TITLE_1_IDX])

[(19, 1), (28, 1), (45, 1), (54, 1), (66, 3), (71, 1), (111, 2), (113, 1), (114, 3), (115, 2), (120, 1), (135, 1), (164, 1), (200, 1), (214, 1), (216, 1), (227, 1), (252, 1), (261, 1), (285, 2), (298, 1), (314, 1), (329, 1), (335, 1), (342, 1), (344, 1), (346, 1), (383, 2), (405, 1), (412, 1), (431, 1), (433, 3), (435, 1), (436, 1), (502, 1), (520, 1), (552, 1), (577, 1), (643, 1), (665, 1), (705, 1), (731, 1), (737, 1), (739, 1), (797, 1), (814, 1)]


In [ ]:
# Review sample document in B0W format
bow_example_doc = bow_corpus[SAMPLE_TITLE_1_IDX]

print('word_index\tword\t\tword_occurences')
for i in range(len(bow_example_doc)):
    print(f'{bow_example_doc[i][0]}\t\t{dictionary[bow_example_doc[i][0]]}\t\t{bow_example_doc[i][1]}')

word_index	word		word_occurences
19		year		1
28		thrill		1
45		come		1
54		better		1
66		chance		3
71		dream		1
111		executive		2
113		home		1
114		life		3
115		producer		2
120		live		1
135		quest		1
164		drama		1
200		kill		1
214		work		1
216		years		1
227		father		1
252		force		1
261		death		1
285		family		2
298		save		1
314		self		1
329		award		1
335		emmy		1
342		win		1
344		action		1
346		give		1
383		bring		2
405		short		1
412		animate		1
431		brother		1
433		second		3
435		mary		1
436		agent		1
502		empire		1
520		writer		1
552		play		1
577		sense		1
643		network		1
665		fall		1
705		later		1
731		children		1
737		scientists		1
739		social		1
797		genius		1
814		abilities		1


### Run TF-IDF on Bag of Words

In [ ]:
# Create tf-idf model object on ‘bow_corpus’
tfidf_model = TfidfModel(bow_corpus)

# Apply transformation to entire corpus
tfidf_corpus = tfidf_model[bow_corpus]

In [ ]:
# Review sample document after tfidf transformation
print('word_index\tword\t\ttfidf_value')
for i in range(len(bow_example_doc)):
    print(f'{bow_example_doc[i][0]}\t\t{dictionary[bow_example_doc[i][0]]}\t\t{tfidf_corpus[SAMPLE_TITLE_1_IDX][i][1]}')

word_index	word		tfidf_value
19		year		0.08524132567596085
28		thrill		0.1355184185569286
45		come		0.07584560445865154
54		better		0.13038327238634131
66		chance		0.3607310335510348
71		dream		0.09509879438249913
111		executive		0.2319914038420434
113		home		0.07839496103162713
114		life		0.17019622955449515
115		producer		0.24048735570068988
120		live		0.06433541819280973
135		quest		0.11489103773046531
164		drama		0.09328104586442616
200		kill		0.11382806907154341
214		work		0.08760623801958849
216		years		0.08543099189084359
227		father		0.09704200113239776
252		force		0.08543099189084359
261		death		0.1045756447768111
285		family		0.13741426684508823
298		save		0.09791767766753402
314		self		0.11714546672405772
329		award		0.11382806907154341
335		emmy		0.12447642516265736
342		win		0.10420258864115733
344		action		0.10817611341982575
346		give		0.11834417215882614
383		bring		0.18606137151679525
405		short		0.13232647913623993
412		animate		0.12604156816261664
431		brother		0.110

### Train LDA model using Bag of Words corpus

In [ ]:
num_cores = multiprocessing.cpu_count()
print(f"Number of CPU cores: {num_cores}")

Number of CPU cores: 2


In [ ]:
%%time

# Train model with Bag of Words corpus
lda_model_bow = LdaMulticore(corpus=bow_corpus, id2word=dictionary, num_topics=10,
                             chunksize=10, passes=10, gamma_threshold=0.001,
                             per_word_topics=True, workers=num_cores, random_state=42)

CPU times: user 7.01 s, sys: 404 ms, total: 7.42 s
Wall time: 9.75 s


In [ ]:
# For each topic, explore the words occuring in that topic and its relative weight
for idx, topic in lda_model_bow.print_topics():
    print('Topic # {}: {}\n'.format(idx, topic))

Topic # 0: 0.066*"star" + 0.040*"find" + 0.039*"start" + 0.034*"heart" + 0.031*"take" + 0.030*"leave" + 0.027*"strange" + 0.026*"executive" + 0.024*"friend" + 0.019*"york"

Topic # 1: 0.052*"bring" + 0.043*"women" + 0.036*"century" + 0.035*"comedy" + 0.035*"series" + 0.031*"popular" + 0.031*"feature" + 0.030*"action" + 0.025*"return" + 0.025*"captain"

Topic # 2: 0.087*"school" + 0.084*"love" + 0.061*"life" + 0.057*"high" + 0.043*"friends" + 0.033*"drama" + 0.031*"turn" + 0.023*"people" + 0.021*"fall" + 0.020*"comedy"

Topic # 3: 0.073*"live" + 0.061*"life" + 0.042*"young" + 0.040*"like" + 0.038*"change" + 0.036*"spirit" + 0.035*"call" + 0.030*"world" + 0.028*"look" + 0.028*"ancient"

Topic # 4: 0.095*"know" + 0.056*"power" + 0.048*"game" + 0.039*"place" + 0.032*"night" + 0.031*"human" + 0.029*"demons" + 0.027*"need" + 0.026*"humans" + 0.025*"chance"

Topic # 5: 0.045*"girl" + 0.038*"year" + 0.036*"fight" + 0.035*"earth" + 0.034*"battle" + 0.029*"paul" + 0.028*"secret" + 0.027*"group" 

In [ ]:
# Alternatively, use show_topics, which gives more options
topics = lda_model_bow.show_topics(num_topics=5, num_words=5, formatted=False)
for topic in topics:
    print(topic)

(2, [('school', 0.086649574), ('love', 0.08426513), ('life', 0.06100714), ('high', 0.05653738), ('friends', 0.043260906)])
(8, [('family', 0.051146593), ('name', 0.030953482), ('world', 0.03045125), ('class', 0.027588738), ('second', 0.027516777)])
(0, [('star', 0.065878585), ('find', 0.039557274), ('start', 0.038802337), ('heart', 0.034334593), ('take', 0.030668136)])
(1, [('bring', 0.05228647), ('women', 0.042836174), ('century', 0.035566986), ('comedy', 0.03506728), ('series', 0.034708235)])
(4, [('know', 0.0949352), ('power', 0.056399383), ('game', 0.04801225), ('place', 0.038717072), ('night', 0.031804703)])


#### Evaluate LDA model trained on Bag of Words corpus

In [ ]:
%%time

# Compute Coherence Score: c_v
coherence_model_lda = CoherenceModel(model=lda_model_bow, texts=processed_docs,
                                     dictionary=dictionary, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('LDA BOW Coherence Score: ', coherence_lda)

LDA BOW Coherence Score:  0.27940797136659934
CPU times: user 640 ms, sys: 27.2 ms, total: 667 ms
Wall time: 669 ms


#### Check LDA BoW model topics on sample document

In [ ]:
# Check sample document
print(processed_docs[SAMPLE_TITLE_1_IDX])

['second', 'chance', 'executive', 'producer', 'writer', 'rand', 'ravich', 'emmy', 'award', 'win', 'executive', 'producer', 'howard', 'gordon', 'come', 'second', 'chance', 'thrill', 'action', 'drama', 'bring', 'life', 'scientists', 'play', 'quest', 'save', 'live', 'seventy', 'year', 'jimmy', 'pritchard', 'philip', 'baker', 'hall', 'shell', 'self', 'drinker', 'womanizer', 'father', 'work', 'family', 'pritchard', 'force', 'resign', 'county', 'sheriff', 'corrupt', 'conduct', 'decade', 'unkind', 'years', 'later', 'kill', 'stumble', 'robbery', 'home', 'agent', 'duval', 'pritchard', 'dekay', 'children', 'death', 'surprisingly', 'short', 'jimmy', 'bring', 'life', 'billionaire', 'tech', 'genius', 'twin', 'mary', 'goodwin', 'dilshad', 'vadsaria', 'brother', 'otto', 'adhir', 'kalyan', 'founder', 'social', 'network', 'empire', 'resurrect', 'younger', 'better', 'version', 'physical', 'abilities', 'dream', 'animate', 'pritchard', 'kazinsky', 'give', 'second', 'chance', 'life', 'repair', 'damage', 'f

In [ ]:
# Convert sample document into the bag-of-words format: list of (token_id, token_count) 2-tuples
sample_bow = dictionary.doc2bow(processed_docs[SAMPLE_TITLE_1_IDX])

In [ ]:
topics = [[word for word, prob in lda_model_bow.show_topic(topicid, topn=5)] \
          for topicid in range(lda_model_bow.num_topics)]

# Get topic distribution for the sample document
topic_distribution = lda_model_bow.get_document_topics(sample_bow)

# Display topic distribution scores for sample document
topic_distribution

[(0, 0.108987935),
 (1, 0.124565065),
 (2, 0.113611095),
 (3, 0.047985654),
 (4, 0.089473724),
 (5, 0.030171875),
 (6, 0.0957007),
 (7, 0.1169494),
 (8, 0.11913087),
 (9, 0.15342367)]

### Train LDA model using TF-IDF corpus

In [ ]:
%%time

# Train model with TFIDF corpus
lda_model_tfidf = LdaMulticore(corpus=tfidf_corpus, id2word=dictionary, num_topics=10,
                               chunksize=10, passes=10, gamma_threshold=0.001,
                               per_word_topics=True, workers=num_cores, random_state=42)

CPU times: user 6.85 s, sys: 346 ms, total: 7.19 s
Wall time: 7.98 s


In [ ]:
# For each topic, explore the words occuring in that topic and its relative weight
for idx, topic in lda_model_tfidf.print_topics():
    print('Topic # {}: {}\n'.format(idx, topic))

Topic # 0: 0.022*"find" + 0.022*"call" + 0.022*"fight" + 0.021*"battle" + 0.021*"game" + 0.019*"young" + 0.019*"like" + 0.019*"force" + 0.018*"team" + 0.017*"spirit"

Topic # 1: 0.070*"world" + 0.028*"space" + 0.027*"character" + 0.023*"chance" + 0.022*"crew" + 0.021*"dead" + 0.021*"evil" + 0.020*"century" + 0.020*"need" + 0.020*"rock"

Topic # 2: 0.049*"school" + 0.038*"love" + 0.032*"high" + 0.028*"friends" + 0.024*"best" + 0.021*"turn" + 0.020*"life" + 0.020*"comedy" + 0.018*"save" + 0.014*"things"

Topic # 3: 0.049*"secret" + 0.039*"demons" + 0.037*"want" + 0.034*"action" + 0.033*"base" + 0.023*"change" + 0.023*"live" + 0.023*"murder" + 0.023*"hero" + 0.023*"look"

Topic # 4: 0.067*"place" + 0.054*"get" + 0.053*"fall" + 0.050*"academy" + 0.034*"escape" + 0.034*"demon" + 0.026*"daughter" + 0.023*"appear" + 0.022*"hand" + 0.022*"earn"

Topic # 5: 0.054*"series" + 0.036*"know" + 0.030*"city" + 0.024*"work" + 0.024*"go" + 0.021*"years" + 0.018*"popular" + 0.017*"stand" + 0.016*"earth" 

In [ ]:
# Alternatively, use show_topics, which gives more options
topics = lda_model_tfidf.show_topics(num_topics=5, num_words=5, formatted=False)
for topic in topics:
    print(topic)

(6, [('meet', 0.031246312), ('time', 0.030032827), ('adventure', 0.028029073), ('learn', 0.026330942), ('past', 0.023808809)])
(7, [('home', 0.025714343), ('girl', 0.023209814), ('life', 0.022754395), ('story', 0.022602852), ('power', 0.022112371)])
(1, [('world', 0.07044401), ('space', 0.028062511), ('character', 0.026929315), ('chance', 0.02295074), ('crew', 0.02167468)])
(0, [('find', 0.021997388), ('call', 0.021637289), ('fight', 0.021570427), ('battle', 0.021042394), ('game', 0.020817209)])
(8, [('live', 0.062219474), ('group', 0.05591377), ('hour', 0.04660397), ('students', 0.039319526), ('serve', 0.03598209)])


#### Evaluate LDA model trained on TF-IDF corpus

In [ ]:
%%time

# Compute Coherence Score: c_v
coherence_model_lda = CoherenceModel(model=lda_model_tfidf, texts=processed_docs,
                                     dictionary=dictionary, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('LDA TFIDF Coherence Score: ', coherence_lda)

LDA TFIDF Coherence Score:  0.3621382830969901
CPU times: user 788 ms, sys: 18.7 ms, total: 806 ms
Wall time: 813 ms


NOTE: Coherence score increased with TF-IDF model

#### Check LDA TF-IDF model topics on sample document

In [ ]:
topics = [[word for word, prob in lda_model_tfidf.show_topic(topicid, topn=5)] \
          for topicid in range(lda_model_tfidf.num_topics)]

# Get topic distribution for the sample document
topic_distribution = lda_model_tfidf.get_document_topics(sample_bow)

# Display topic distribution scores for sample document
topic_distribution

[(0, 0.078657135),
 (1, 0.11842487),
 (2, 0.15040219),
 (3, 0.02100824),
 (4, 0.019296529),
 (5, 0.09158148),
 (6, 0.05007462),
 (7, 0.2769027),
 (8, 0.09544501),
 (9, 0.09820724)]